# Validation Notebook for SSLimPy halomodel 

In this notebook we will study the different components of the LIM halo model computation and compare them to the Colossus results

## Initial Setup

In [ ]:
import os
import sys

sys.path.append("../")

envkey = "OMP_NUM_THREADS"
# Set this environment variable to the number of available cores in your machine,
# to get a fast execution of the Einstein Boltzmann Solver
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))
os.environ[envkey] = str(12)
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
from copy import copy, deepcopy
import numpy as np
import seaborn
from getdist.gaussian_mixtures import GaussianND
from getdist import plots
import matplotlib.patches as mpatches

In [ ]:
# seaborn.set_theme(rc={'axes.edgecolor': 'black', 'xtick.color': 'black', 'ytick.color': 'black',})
import niceplots.utils as nicepl
nicepl.initPlot()

In [ ]:
from scipy.signal import find_peaks
from scipy.interpolate import UnivariateSpline

In [ ]:
Cs = seaborn.color_palette("Paired")
Cs

In [ ]:
Cp = seaborn.color_palette("colorblind")
Cp

## Choose model parameters and save them in dictionaries

In [ ]:
settings = {
    "code":"class", # The Einstein--Boltzman solver that should be used
    "do_RSD" : False, # If RSD should be considerd
    "nonlinearRSD" : False, # If you want to add FOG to the RSD
    "QNLpowerspectrum": False, # Use dewiggled power spectrum (vlasov approximation of nonlinear structure formation)
    "FoG_damp" : "ISTF_like", # The particular parametrization for the FOG. Check PowerSpectrum for the full list
    "halo_model_PS" : True, # If the cosmological shotnoise should be computed from the halo model 
    "output" : ["Power spectrum", "Covariance"], # What output one wants (here power spectrum and Gaussian covariance only)
    "kmin": 1e-4 * u.Mpc**-1,
    "kmax": 50 * u.Mpc**-1,
    "nk": 200,
}

cosmodict={
    "h": 0.677,
    "Omegam": 0.309167,
    "Omegab": 0.04903,
    "sigma8":0.8222,
    "ns":0.96824,
    "mnu":0.06,
    "Neff":3.044,
}

# Parameters that enter your halo model. Typically they are not changed but you could
halodict={
    "halo_tracer" : "clustering", # Computes all halo quantities from the matter field - neutrinos
    "hmf_model": "ST", # Sheth--Tormann halo mass function
    "concentration": "Diemer19", # Diemer19 halo concentration relation
    "bias_model": "ST99",
}

# Parameters for the Survey specifications
z = np.array([1])
def get_specs(nu):
        nuObs = nu / (z + 1)
        DeltaDeltanu = 0.05143

        surveyspecs = {
                "Tsys_NEFD": 0 * u.uK, #System temperature for instrumental shotnoise
                "Nfeeds": 19,
                "tobs": 1300 * u.h,
                "nD": 1, # Observational parameters
                "beam_FWHM": 0. * u.arcmin,
                "nu":  nu,
                "nuObs": nuObs, # Observed Frequency
                "Delta_nu": DeltaDeltanu * nuObs, # Frequency Bin
                "dnu": 0. * u.MHz, # Spectrograph resolution
                "Omega_field": 18.64 * u.deg**2, # Angular size of survey
        }
        return surveyspecs

astrodict_CII={
    "model_type": "ML",
    "model_name": "SilvaCII",
    "model_par": {
        "a": 0.8475,
        "b": 7.2203,
        "SFR_file": "sfr_release.dat",
        "do_quench": False,
    },
    "sigma_scatter" : 0.37,
    "meanperserve_scatter": False,
}

nu = 1.897 * u.THz
surveyspecs_CII = get_specs(nu)

## Compute instance of SSLimPy to work with

In [ ]:
# Import Main modules. This might take some time as some functions compile before time
from SSLimPy.interface import sslimpy
from SSLimPy.interface import updater
from SSLimPy.cosmology import cosmology
from SSLimPy.cosmology import halo_model
from SSLimPy.cosmology import astro
from SSLimPy.LIMsurvey import power_spectrum
from SSLimPy.LIMsurvey import covariance
from SSLimPy.LIMsurvey import higher_order
from SSLimPy.LIMsurvey import ingredients_T0

In [ ]:
from SSLimPy.utils.utils import *

In [ ]:
myssl = sslimpy.SSLimPy(
    settings_dict=settings,
    cosmopars=cosmodict,
    halopars=halodict,
    astropars=astrodict_CII,
    obspars_dict=surveyspecs_CII,
)

In [ ]:
cosmo = myssl.current_cosmology
halo = myssl.current_halomodel
myyastro = myssl.current_astro

In [ ]:
from colossus.cosmology import cosmology
from colossus.lss import mass_function
from colossus.lss import bias
from colossus.halo import mass_so
from colossus.halo import concentration

In [ ]:
Colossus_cosmo = cosmology.setCosmology(
    "mycosmo",
    {
        "flat":True,
        "H0": cosmodict["h"] * 100,
        "Ob0": cosmodict["Omegab"],
        "sigma8": cosmodict["sigma8"],
        "ns": cosmodict["ns"],
        "Om0": cosmodict["Omegam"],
    }
)

In [ ]:
# all lenghts and masses in Colossus are rescaled by h
h = cosmo.h()


In [ ]:
z = np.linspace(0, 4)
cDa = Colossus_cosmo.angularDiameterDistance(z) / h
sDa = cosmo.angdist(z)

plt.semilogy(z, cDa, c=Cs[0], label="Colossus")
plt.semilogy(z, sDa, ls="--", c=Cs[1], label="SSLimPy")
plt.legend()
plt.xlabel("redshift $z$")
plt.ylabel(r"$D_\mathrm{A}\,[\mathrm{Mpc}]$")
plt.tight_layout()

In [ ]:
k = np.geomspace(1e-3, 1, 50) * u.Mpc**-1
cPk = Colossus_cosmo.matterPowerSpectrum(k.value / h, 0) / h**3
sPk = cosmo.matpow(k, 0)

plt.loglog(k, cPk, c=Cs[0], label="Colossus")
plt.loglog(k, sPk, ls="--", c=Cs[1], label="SSLimPy")
plt.legend()
plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$P(k)\,[\mathrm{Mpc}^3]$")
plt.tight_layout()

In [ ]:
R = np.geomspace(1, 100, 20) * u.Mpc
csR = Colossus_cosmo.sigma(R.value * h, 1.0)
ssR = halo.sigmaR_of_z(R, 1.0)

plt.loglog(R, csR, c=Cs[0], label="Colossus")
plt.loglog(R, ssR, ls="--", c=Cs[1], label="SSLimPy")

plt.legend()
plt.xlabel(r"$R\,[\mathrm{Mpc}]$")
plt.ylabel(r"$\sigma_R$")
plt.tight_layout()

In [ ]:
cmf=mass_function.massFunction(csR, 1.0, q_in="sigma", q_out="dndlnM", model="sheth99")

MofR = linear_interpolate(halo.R.value, halo.M.value, R.value) * halo.M.unit 
smf=halo.halomassfunction(MofR, 1.00) * MofR 

plt.loglog(csR, cmf * h**3, c=Cs[0], label="Colossus")
plt.loglog(ssR, smf, ls="--", c=Cs[1], label="SSLimPy")
plt.xlabel(r"$\sigma_R$")
plt.ylabel(r"$\mathrm{d}N\,/\,\mathrm{dlog}M\,[\mathrm{Mpc}^{-3}]$")

In [ ]:
cb1 = bias.haloBiasFromNu(halo.delta_crit/ csR, 1.0, model="sheth01", mdef="1m")
sb1 = halo.get_bias(MofR, 1.0, beta=1)
plt.loglog(halo.delta_crit/ csR, cb1, c=Cs[0], label="Colossus")
plt.loglog(halo.delta_crit/ ssR, sb1, ls="--", c=Cs[1], label="SSLimPy")
plt.legend()
plt.xlabel(r"peak height $\nu$")
plt.ylabel(r"halo bias $b_1$")

In [ ]:
M = np.geomspace(1e10, 1e15) * u.Msun
ccM = concentration.concentration(M.value * h, "200c", 0.0, "diemer19", "mean")
scM = halo.concentration(M, 0.0)

plt.semilogx(M, ccM, c=Cs[0], label="Colossus")
plt.semilogx(M, scM, c=Cs[1], label="SSLimPy", ls="--")
plt.legend()
plt.xlabel(r"$M\,[M_\odot]$")

In [ ]:
k = np.geomspace(1e-3, 10, 100) * u.Mpc**-1
I0 = halo.Ihalo(1.0, k, p=1, scale=(2,), beta="b0")
I1 = 1# halo.Ihalo(1.0, k, p=1, scale=(1,), beta="b1")
plin = cosmo.matpow(k, 1.0, nonlinear=False, tracer="clustering")
phalo = I1**2 * plin + I0
pnl = cosmo.matpow(k, 1.0, nonlinear=True, tracer="clustering")

plt.loglog(k, plin, "k--")
plt.loglog(k, pnl, c=Cs[0], label="HMCode")
plt.loglog(k, phalo, c=Cs[1], label="SSLimPy")
plt.legend()
plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$P(k)\,[\mathrm{Mpc}^3]$")

In [ ]:
colors = iter(Cp)

z = np.linspace(0, 5)
b1 = myyastro.bavg("b1", z, power=1)
b2 = myyastro.bavg("b2", z, power=1)
b3 = myyastro.bavg("b3", z, power=1)
D = cosmo.growth_factor(1e-3 * u.Mpc**-1, z)

for b, l in zip([b1 * D, b2 * D**2 / 2, b3 * D**3 / 6], [r"$b_1\,D$", r"$\frac{b_2}{2}\,D^2$", r"$\frac{b_3}{6}\,D^3$"]):
    c = next(colors)
    plt.plot(z, b, label=l, color=c)
    plt.plot(z, -b, color=c, ls= "--")

plt.legend()
plt.xlabel("redshift $z$")
plt.ylabel(r"\textbar bias\textbar")
plt.ylim(0, 2)
plt.tight_layout()
plt.title("[CII]")
plt.savefig("output/bias_evolution.pdf")

In [ ]:
cosmo.h()

In [ ]:
R = np.geomspace(1e-1, 1e4) * u.Mpc

def R_to_k(R):
    return (np.pi / R / 2) / 0.677

def k_to_R(k):
    return np.pi / k / 2

# Choose redshifts
z_values = [1.0, 2.5]#, 3.5, 4.5]

fig, axes = plt.subplots(1, 2, figsize=(6, 3), sharex=True, sharey=True)
colors = iter(Cp)  # keep your color palette

for ax, z in zip(axes.flat, z_values):
    # Compute factors
    sigmaR1 = halo.sigmaR_of_z(R, z)
    b1fac = sigmaR1
    b2fac = np.sqrt(2) * sigmaR1**2
    b3fac = np.sqrt(15) * sigmaR1**3

    # Biases
    b1 = myyastro.bavg("b1", z, power=1)
    b2 = myyastro.bavg("b2", z, power=1)
    b3 = myyastro.bavg("b3", z, power=1)

    #compute the R_NL
    Rnl = R[
        np.argmin(
            np.abs(sigmaR1-1)
        )
    ]

    Rperturb = R[
        np.argmin(
                ((np.abs(b2 * b2fac / 2) / np.abs(b3 * b3fac / 6))-1)**2
            )
    ]
    print(np.abs(b2 * b2fac / 2)[-1])
    print(Rperturb)
    # Reset colors for each subplot so curves stay consistent
    color_iter = iter(Cp)

    # Plot
    for b, l in zip(
        [b1 * b1fac, b2 * b2fac / 2, b3 * b3fac / 6],
        [r"$b_1\,\sigma_R$", r"$\frac{b_2}{2}\,\sqrt{2}\,\sigma_R^2$", r"$\frac{b_3}{6}\,\sqrt{15}\,\sigma_R^3$"]
    ):
        c = next(color_iter)
        ax.loglog(R, b, label=l, color=c)
        ax.loglog(R, -b, color=c, ls="--")

    # Labels and secondary axis
    ax.set_xlabel(r"Smoothing scale $R\,[\mathrm{Mpc}]$")
    ax.set_ylabel(r"rescaled bias")
    ax.tick_params(axis="x", which="both", top=False)

    ax.axvspan(0, Rnl.value, color="lightcoral", alpha=0.2, hatch="/")
    if Rnl < Rperturb:
        ax.axvspan(Rnl.value, Rperturb.value, color="plum", alpha=0.2, hatch="/")

    ax.hlines(1, min(R.value), max(R.value), color="k", ls=":")

    secax = ax.secondary_xaxis("top", functions=(R_to_k, k_to_R))
    secax.set_xlabel(r"Wavenumber $k\,[h\,\mathrm{Mpc}^{-1}]$")

    # Add z label inside subplot
    ax.text(0.95, 0.95, f"$z$ = {z}", transform=ax.transAxes,
            ha="right", va="top",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.7))

# Legend only once (upper-left panel)
axes[0].legend()

# After plotting the spans, define dummy handles for the vspan legend
span1 = mpatches.Patch(color="lightcoral", alpha=0.2, hatch="/", label=r"non-linear scale")
span2 = mpatches.Patch(color="plum", alpha=0.2, hatch="/", label=r"non-Gaussian scale")

# Curve legend in first subplot (axes[0])
axes[0].legend()

# Vspan legend in second subplot (axes[1])
axes[1].legend(handles=[span1, span2])
plt.suptitle("[CII]")
plt.tight_layout()
plt.savefig("output/pertive_scale.pdf")

In [ ]:
dNdlnM = M * halo.halomassfunction(M, 1)
LofM = myyastro.massluminosityfunction(M, 1)

b1ofM = halo._bias_function.b1(M, 1, halo.delta_crit)
b2ofM = halo._bias_function.b2(M, 1, halo.delta_crit)
b3ofM = halo._bias_function.b3(M, 1, halo.delta_crit)

In [ ]:

for b, c, l in zip([b1ofM, b2ofM, b3ofM], [Cp[0], Cp[1], Cp[2]], ["$b_1$", "$b_2$", "$b_3$"]): 
    plt.loglog(M, b * dNdlnM * LofM, c=c, label=l)
    plt.loglog(M, -b * dNdlnM * LofM, c=c, ls="--")
plt.legend()
plt.xlim(1, 5e16)
plt.ylim(1e-15, 1e6)

In [ ]:
k = cosmo.k

In [ ]:
a  = np.linspace(0.9, 1.0, 5)
z = np.linspace(0, 4)

b1_of_z = []
b2_of_z = []
for ai in a:
    atroi = deepcopy(astrodict)    
    atroi["model_par"]["a"] = ai
    current_astro = updater.update_astro(myssl.current_astro, myssl.fiducialcosmoparams, myssl.fiducialhaloparams, atroi, myssl.fiducialspecparams, myssl.cfg)
    b1_of_z.append(current_astro.bavg("b1", z, 1))
    b2_of_z.append(current_astro.bavg("b2", z, 1))


In [ ]:
fig, ax = plt.subplots()

# Plotting with labels only once for b1 and b2
for i in range(5):
    alpha_val = (1 + i) / 6
    line1, = ax.plot(z, b1_of_z[i], alpha=alpha_val, c="red")
    line2, = ax.plot(z, b2_of_z[i], alpha=alpha_val, c="blue", ls="--")
    mulligen, = ax.plot([], [], alpha=alpha_val, c="k", label=f"a = {a[i]:.2f}")

legend1 = ax.legend(title="SFR shape", loc="upper left")
ax.add_artist(legend1)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color="red", lw=2, label="$b_1$"),
    Line2D([0], [0], color="blue", lw=2, linestyle="--", label="$b_2$")
]
ax.legend(handles=legend_elements, title="Halo bias", loc="lower right")

ax.tick_params(axis='both', direction="in", which='major', labelsize=12, width=1.5, length=7,)
ax.tick_params(axis='both', direction="in", which='minor', labelsize=10, width=1., length=2,)
ax.minorticks_on()

plt.xlabel("$z$")
plt.ylabel("Bias")
plt.title("Bias vs Redshift for Different a Values")
plt.show()

In [ ]:
pobs = myssl.compute(
    myssl.fiducialcosmoparams,
    myssl.fiducialhaloparams,
    myssl.fiducialastroparams,
    myssl.fiducialspecparams,
    output=["Power spectrum"]
)["Power spectrum"]

In [ ]:
NGcov = covariance.nonGuassianCov(pobs)

In [ ]:
an1, cn1 = NGcov.fftLog_Pofk.get_power_and_coef()

In [ ]:
plt.semilogy(an1.imag, np.abs(cn1.real))

# Comparasion of CII vs matter only

In [ ]:
fid_cosmopars = myssl.current_cosmology.cosmopars
fid_halopars = myssl.current_halomodel.haloparams

In [ ]:
pobs_settings = {
    "kmin" : 1e-3 * u.Mpc**-1,
    "kmax" : 5 * u.Mpc**-1,
    "nk" : 100,
    "k_kind": "log",
    "nmu" : 128,
}

pobs_1 = myssl.compute(fid_cosmopars, fid_halopars, astrodict, surveyspecs, pobs_settings=pobs_settings, output=["Power spectrum"])["Power spectrum"]
pobs_2 = myssl.compute(fid_cosmopars, fid_halopars, astrodict2, surveyspecs, pobs_settings=pobs_settings, output=["Power spectrum"])["Power spectrum"]

In [ ]:
plt.loglog(pobs_1.k, pobs_1.Pk_0bs / np.max(pobs_1.Pk_0bs), label="CII")
plt.loglog(pobs_1.k, pobs_2.Pk_0bs / np.max(pobs_2.Pk_0bs), label="matter")
plt.xlim(1e-3, 1)
plt.ylim(1e-2, 1.2)

In [ ]:
# Biasblot
z = np.linspace(0, 8, num=200)

fig, axs = plt.subplots(2, 1, figsize=(12, 8))

b1_of_z = pobs_1.astro.bavg("b1", z, power=1)
b2_of_z = pobs_1.astro.bavg("b2", z, power=1)
b3_of_z = pobs_1.astro.bavg("b3", z, power=1)

axs[0].semilogy(z, b1_of_z, label="$b_1$", c=Cs[1])
axs[0].semilogy(z, b2_of_z, label="$b_2$", c=Cs[3])
axs[0].semilogy(z, b3_of_z, label="$b_3$", c=Cs[5])
axs[1].semilogy(z, -b1_of_z,c=Cs[1])
axs[1].semilogy(z, -b2_of_z,c=Cs[3])
axs[1].semilogy(z, -b3_of_z,c=Cs[5])

b1_of_z = pobs_2.astro.bavg("b1", z, power=1)
b2_of_z = pobs_2.astro.bavg("b2", z, power=1)
b3_of_z = pobs_2.astro.bavg("b3", z, power=1)

axs[0].semilogy(z, b1_of_z, c=Cs[1-1], ls="--")
axs[0].semilogy(z, b2_of_z, c=Cs[3-1], ls="--")
axs[0].semilogy(z, b3_of_z, c=Cs[5-1], ls="--")
axs[1].semilogy(z, -b1_of_z,c=Cs[1-1], ls="--")
axs[1].semilogy(z, -b2_of_z,c=Cs[3-1], ls="--")
axs[1].semilogy(z, -b3_of_z,c=Cs[5-1], ls="--")


axs[0].legend(fontsize=16)

axs[1].semilogy([],[],ls="-", color="k", label="Luminoisity weight")
axs[1].semilogy([],[],ls="--", color="gray", label="Halo mass weight")
axs[1].legend(fontsize=16)

axs[1].set_xlabel("$z$", fontsize=16)
fig.supylabel(r"$\langle$ bias $\rangle$ for [CII]", fontsize=16)

yticks = axs[0].get_yticks()
yticklabels = [r"$10^{{{}}}$".format(int(np.log10(ytick))) for ytick in yticks]
axs[0].set_yticks(yticks, labels=yticklabels)
axs[0].set_ylim(1e-1, 1e2)

yticks = axs[1].get_yticks()
yticklabels = [r"$-10^{{{}}}$".format(int(np.log10(ytick))) for ytick in yticks]
axs[1].set_yticks(yticks, labels=yticklabels)
axs[1].set_ylim(1e-1, 5)
axs[1].invert_yaxis()

In [ ]:
cosmo = pobs_1.cosmology

In [ ]:
NG_cov_CII = covariance.nonGuassianCov(pobs_1)

In [ ]:
CII_T_1h = NG_cov_CII.integrate_1h()
CII_T_2h = NG_cov_CII.integrate_2h()
CII_T_3h = NG_cov_CII.integrate_3h()
CII_T_4h = NG_cov_CII.integrate_4h()

In [ ]:
CII_T_1h.shape

In [ ]:
fig, axs = plt.subplots(2, 2, sharex=True, sharey=True, figsize=(10, 10))
indicies = [15, 35, 65, 85]

for i, ax in zip(indicies, axs.flatten()):
    ax.loglog(pobs_1.k, (CII_T_1h[:, i, 0]))
    ax.loglog(pobs_1.k, (CII_T_2h[:, i, 0]))
    ax.loglog(pobs_1.k, (CII_T_3h[:, i, 0]))
    ax.loglog(pobs_1.k, (CII_T_4h[:, i, 0]))
fig.tight_layout()

In [ ]:
plt.loglog(pobs_1.k, np.diag(CII_T_1h[:, :, 0]))
plt.loglog(pobs_1.k, np.diag(CII_T_2h[:, :, 0]))
plt.loglog(pobs_1.k, np.diag(CII_T_3h[:, :, 0]))
plt.loglog(pobs_1.k, np.diag(CII_T_4h[:, :, 0]))

In [ ]:
Vfield = NG_cov_CII.survey_specs.Vfield()

In [ ]:
NG_cov = (CII_T_1h + CII_T_2h + CII_T_3h + CII_T_4h) / Vfield

In [ ]:
G_cov_CII = covariance.Covariance(pobs_1)
G_cov = G_cov_CII.gaussian_cov()

G_cov = G_cov[:, 0, 0, 0]
NG_cov = NG_cov[:, :, 0]

In [ ]:
Cov = np.diag(G_cov) + NG_cov
Corr = Cov / np.sqrt(np.outer(np.diag(Cov), np.diag(Cov)))

fig, ax = plt.subplots()

cmap_choice = 'viridis' 


my_cmap = plt.get_cmap(cmap_choice)
my_cmap.set_over("red")
my_cmap.set_under("white")

cax = ax.imshow(
    Corr,
    cmap=my_cmap,
    vmax=1,
    vmin=0,
)

cbar = fig.colorbar(cax)
cbar.set_label('Corrlation')
ax.set_xlabel(r'$k_2\,[\mathrm{Mpc}^{-1}]$')
ax.set_ylabel(r'$k_1\,[\mathrm{Mpc}^{-1}]$')

# Suppose q_vals represents the underlying log-spaced physical values (e.g., logspace)
q_vals = pobs_1.k.value

log_ticks = np.arange(np.ceil(np.log10(q_vals[0])), np.floor(np.log10(q_vals[-1])) + 1)
tick_values = 10 ** log_ticks

# Find the indices in q_vals closest to these tick_values
tick_indices = [np.argmin(np.abs(q_vals - val)) for val in tick_values]
tick_labels = [f"$10^{{{int(t)}}}$" for t in log_ticks]

# Apply to plot
ax.set_xticks(tick_indices)
ax.set_xticklabels(tick_labels)
ax.set_yticks(tick_indices)
ax.set_yticklabels(tick_labels)

plt.show()

In [ ]:
mTb = NG_cov_CII.astro.Tbavg(1, pobs_1.z, 1)


In [ ]:
k = NG_cov_CII.k
Pk = NG_cov_CII.Pk

In [ ]:
Pt = np.sqrt(CII_T_4h[:, :, 0] / Vfield / mTb**4 / np.outer(Pk, Pk)).to(1).value

In [ ]:
np.argmin(np.abs(k-0.92 * NG_cov_CII.astro.Mpch**-1))

In [ ]:
fig, axs = plt.subplots(2, 3, sharex=True, sharey=True, figsize=(16,11))

indx = [30, 50, 62, 68, 71, 74]
for id, ax in zip(indx, axs.flatten()):
    ax.semilogx(k, Pt[:, id], c=Cs[1])
    ax.scatter([],[], label=r"$k_1:{{{:.2}}}".format(k[id].value)+"\:\mathrm{Mpc}^{-1}$", alpha=0.0)
    ax.legend(frameon=False, loc="lower left", fontsize=16)
    ax.set_xlim(1e-2, 1)
    ax.tick_params(axis="both", which="major", labelsize=16)

axs[0,0].set_ylabel(r"$\sqrt{\mathrm{Cov}_\mathrm{4h}(k_1, k_2)\,/\,P(k_1)\,P(k_2)}$", fontsize=16)
axs[1,0].set_ylabel(r"$\sqrt{\mathrm{Cov}_\mathrm{4h}(k_1, k_2)\,/\,P(k_1)\,P(k_2)}$", fontsize=16)

axs[1,0].set_xlabel(r"$k_2\,[\mathrm{Mpc}^{-1}]$", fontsize=16)
axs[1,1].set_xlabel(r"$k_2\,[\mathrm{Mpc}^{-1}]$", fontsize=16)
axs[1,2].set_xlabel(r"$k_2\,[\mathrm{Mpc}^{-1}]$", fontsize=16)

fig.tight_layout()
fig.subplots_adjust(hspace=0.1, wspace=0.1)

In [ ]:
NG_cov_MO = covariance.nonGuassianCov(pobs_2)
MO_T_1h = NG_cov_MO.integrate_1h()
MO_T_2h = NG_cov_MO.integrate_2h()
MO_T_3h = NG_cov_MO.integrate_3h()
MO_T_4h = NG_cov_MO.integrate_4h()

In [ ]:
NG_cov_CII.fftLog_Pofk.nu.shape

In [ ]:
NG_cov_CII.cfg.settings

In [ ]:
k = NG_cov_MO.k
fig, axs = plt.subplots(1, 2, sharex=True, figsize=(20, 8))
axs[0].loglog(k, np.diag(MO_T_1h[:,:,0]), label="1h")
axs[0].loglog(k, np.diag(MO_T_2h[:,:,0]), label="2h")
axs[0].loglog(k, np.diag(MO_T_3h[:,:,0]), label="3h")
axs[0].loglog(k, np.diag(MO_T_4h[:,:,0]), "r", label="4h")
axs[0].legend()

axs[1].loglog(k, np.diag(CII_T_1h[:,:,0]), label="1h")
axs[1].loglog(k, np.diag(CII_T_2h[:,:,0]), label="2h")
axs[1].loglog(k, np.diag(CII_T_3h[:,:,0]), label="3h")
axs[1].loglog(k, np.diag(CII_T_4h[:,:,0]), "r", label="4h")
axs[1].loglog(k, np.diag(-CII_T_4h[:,:,0]), "r--", label="4h")


In [ ]:
CII_b1 = pobs_1.astro.Thalo(pobs_1.z, k, p=1, scale=(1,), beta="b1") / pobs_1.astro.Tavg(pobs_1.z)
CII_b2 = pobs_1.astro.Thalo(pobs_1.z, k, p=1, scale=(1,), beta="b2") / pobs_1.astro.Tavg(pobs_1.z)
CII_b3 = pobs_1.astro.Thalo(pobs_1.z, k, p=1, scale=(1,), beta="b3") / pobs_1.astro.Tavg(pobs_1.z)

In [ ]:
MO_b1 = pobs_2.astro.Thalo(pobs_2.z, k, p=1, scale=(1,), beta="b1") / pobs_2.astro.Tavg(pobs_2.z)
MO_b2 = pobs_2.astro.Thalo(pobs_2.z, k, p=1, scale=(1,), beta="b2") / pobs_2.astro.Tavg(pobs_2.z)
MO_b3 = pobs_2.astro.Thalo(pobs_2.z, k, p=1, scale=(1,), beta="b3") / pobs_2.astro.Tavg(pobs_2.z)

In [ ]:
print("MO b1:", MO_b1[0], "\t CII b1:", CII_b1[0])
print("MO b2:", MO_b2[0], "\t CII b2:", CII_b2[0])
print("MO b3:", MO_b3[0], "\t CII b3:", CII_b3[0])

In [ ]:
M = pobs_1.halomodel.M
dc = pobs_1.halomodel.delta_crit
z = pobs_1.z

hmf  = pobs_1.halomodel.halomassfunction(M, z)

b1 = pobs_1.halomodel._bias_function.b1(M, z, dc)
b2 = pobs_1.halomodel._bias_function.b2(M, z, dc)
b3 = pobs_1.halomodel._bias_function.b3(M, z, dc)

LofM = pobs_1.astro.massluminosityfunction(M, z)
Moverho = pobs_2.astro.massluminosityfunction(M, z)


In [ ]:
fig, axs = plt.subplots(1, 2, sharex=True, figsize=(20, 8))
#axs[1].loglog(M,M * hmf * LofM**4)
axs[1].loglog(M,M * hmf * LofM)
# axs[0].loglog(M,M * hmf * Moverho**4)
axs[0].loglog(M,M * hmf * Moverho)
axs[0].grid()
axs[1].grid()

In [ ]:
fig, axs = plt.subplots(1, 2, sharex=True, figsize=(20, 8))
colors = iter(Cs)

norm1_integ = hmf * LofM
norm1 = np.trapz(norm1_integ * M, np.log(M.value))
norm2_integ = hmf * Moverho
norm2 = np.trapz(norm2_integ * M, np.log(M.value))

biases = [b1, b2, b3]
names = ["b1", "b2", "b3"]
for name, b in zip(names, biases):
    c=next(colors)
    axs[0].loglog(M, -M * Moverho * hmf * b / norm2, c=c)
    axs[1].loglog(M, -M * LofM * hmf * b / norm1, c=c)
    c=next(colors)
    axs[0].loglog(M, M * Moverho * hmf * b / norm2, c=c, label=name)
    axs[1].loglog(M, M * LofM * hmf * b / norm1, c=c)
axs[0].legend()


In [ ]:
fig, axs = plt.subplots(1, 2, sharex=True, figsize=(20, 8))
colors = iter(Cs)

norm1_integ = hmf * LofM**2
norm1 = np.trapz(norm1_integ * M, np.log(M.value))
norm2_integ = hmf * Moverho**2
norm2 = np.trapz(norm2_integ * M, np.log(M.value))

biases = [b1, b2, b3]
names = ["b1", "b2", "b3"]
for name, b in zip(names, biases):
    c=next(colors)
    axs[0].loglog(M, -M * Moverho**2 * hmf * b / norm2, c=c)
    axs[1].loglog(M, -M * LofM**2 * hmf * b / norm1, c=c)
    c=next(colors)
    axs[0].loglog(M, M * Moverho**2 * hmf * b / norm2, c=c, label=name)
    axs[1].loglog(M, M * LofM**2 * hmf * b / norm1, c=c)
axs[0].legend()

In [ ]:
CII_gaussian_cov = covariance.Covariance(pobs_1).gaussian_cov()[:, 0, 0, 0]
MO_gaussian_cov = covariance.Covariance(pobs_2).gaussian_cov()[:, 0, 0, 0]

In [ ]:
CII_nongaussian_cov = ((CII_T_1h + CII_T_2h + CII_T_3h+ CII_T_4h) / NG_cov_CII.survey_specs.Vfield())[:, :, 0]
MO_nongaussian_cov = ((MO_T_1h + MO_T_2h + MO_T_3h+ MO_T_4h) / NG_cov_MO.survey_specs.Vfield())[:, :, 0]


In [ ]:
CII_total_cov = CII_nongaussian_cov + np.diag(CII_gaussian_cov)
MO_total_cov = MO_nongaussian_cov + np.diag(MO_gaussian_cov)

In [ ]:
dCII = np.diag(CII_total_cov)
# corr_CII = CII_total_cov / np.sqrt(np.outer(dCII, dCII))
dMO = np.diag(MO_total_cov)
corr_MO = MO_total_cov / np.sqrt(np.outer(dMO, dMO))

In [ ]:
corr_MO

In [ ]:
plt.imshow(corr_MO)